In [45]:
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader

def MNIST_loader():
    training_data = datasets.MNIST('./data/MNIST/',
                                 download=True,
                                 train=True,transform=ToTensor())
    test_data = datasets.MNIST('./data/MNIST/',
                                    download=True,
                                    train=False,transform=ToTensor())
    train_dataloader = DataLoader(training_data , batch_size = 32 , shuffle = True)
    test_dataloader = DataLoader(test_data , batch_size = 32 , shuffle = True)

    return train_dataloader,test_dataloader


def CIFAR10_loader():
    training_data = datasets.CIFAR10('./data/CIFAR10/',
                                 download=True,
                                 train=True,transform=ToTensor())
    test_data = datasets.CIFAR10('./data/CIFAR10/',
                                    download=True,
                                    train=False,transform=ToTensor())
    train_dataloader = DataLoader(training_data , batch_size = 256 , shuffle = True)
    test_dataloader = DataLoader(test_data , batch_size = 256 , shuffle = True)

    return train_dataloader,test_dataloader


In [46]:
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.cluster import KMeans

class modified_linear(nn.Linear):
    def __init__(self, in_features, out_features,mode = 'normal', bias = True, device=None, dtype=None):
        super().__init__(in_features, out_features, bias, device, dtype)
        self.mode = mode
        self.register_buffer("mask", torch.ones_like(self.weight))
        self.register_buffer(
            "assignments",
            torch.zeros_like(self.weight, dtype=torch.long)  
        )
        self.hashtable = None 
    
    def prune(self,threshold):
        self.mode = 'prune'
        new_mask = (self.weight.abs() > threshold).float()   
        self.mask.mul_(new_mask)
        print("pruned!!")
        return
    
    def quantize(self,k):
        self.mode = 'quantize'
        matrix = self.weight.detach().cpu().numpy()
        kmeans = KMeans(
                n_clusters=k,
                random_state=42,
                n_init=10
            )
        kmeans.fit(matrix.reshape(-1,1))
        assignments =torch.from_numpy(kmeans.labels_).reshape(self.weight.shape).long()
        self.assignments.copy_(assignments)
        self.hashtable = nn.Parameter(
            torch.from_numpy(
                kmeans.cluster_centers_.reshape(-1)
            ).to(self.weight.device, self.weight.dtype)
        )
        self.weight.requires_grad = False
        return
    
    def forward(self, input):
        if(self.mode == 'normal'):
            W = self.weight 
        elif(self.mode == 'prune'):
            W = (self.weight * self.mask)
        elif(self.mode == 'quantize'):
            W = self.hashtable[self.assignments] * self.mask  
        return F.linear(input,W,self.bias)

class modified_conv2d(nn.Conv2d):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        stride=1,
        padding=0,
        dilation=1,
        groups=1,
        bias=True,
        padding_mode="zeros",
        mode="normal",
        device=None,
        dtype=None,
    ):
        super().__init__(
            in_channels,
            out_channels,
            kernel_size,
            stride,
            padding,
            dilation,
            groups,
            bias,
            padding_mode,
            device=device,
            dtype=dtype,
        )
        self.mode = mode
        self.register_buffer("mask", torch.ones_like(self.weight))
        self.register_buffer(
            "assignments",
            torch.zeros_like(self.weight, dtype=torch.long),
        )
        self.hashtable = None 

    def prune(self, threshold):
        self.mode = "prune"
        new_mask = (self.weight.abs() > threshold).float()
        self.mask.mul_(new_mask)  
        print("pruned!!")
        return

    def quantize(self, k):
        self.mode = "quantize"
        matrix = self.weight.detach().cpu().numpy()
        kmeans = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=10,
        )
        kmeans.fit(matrix.reshape(-1, 1))
        assignments = torch.from_numpy(
            kmeans.labels_
        ).reshape(self.weight.shape).long()
        self.assignments.copy_(assignments)
        self.hashtable = nn.Parameter(
            torch.from_numpy(
                kmeans.cluster_centers_.reshape(-1)
            ).to(self.weight.device, self.weight.dtype)
        )
        self.weight.requires_grad = False
        return

    def forward(self, input):
        if self.mode == "normal":
            W = self.weight
        elif self.mode == "prune":
            W = self.weight * self.mask
        elif self.mode == "quantize":
            W = self.hashtable[self.assignments] * self.mask
        return F.conv2d(
            input,
            W,
            self.bias,
            self.stride,
            self.padding,
            self.dilation,
            self.groups,
        )

In [47]:
from torch import nn

class mnist_model(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.classifier = nn.Sequential(
            modified_linear(28 * 28, 256),
            nn.ReLU(),
            modified_linear(256, 512),
            nn.ReLU(),
            modified_linear(512, 256),
            nn.ReLU(),
            modified_linear(256, 10)
        )
        
    def forward(self,x):
        x = self.flatten(x)
        x = self.classifier(x)
        return x
    
    def prune(self,threshold):
        for module in self.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                module.prune(threshold)
    
    def quantize(self,k):
        for module in self.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                module.quantize(k)

class SmallCIFARNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            modified_conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            modified_conv2d(32, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2), 

            modified_conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            modified_conv2d(64, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  
            
            modified_conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2), 
        )
        self.flatten = nn.Flatten()
        self.softmax = nn.Softmax(dim = 1)
        self.classifier = nn.Sequential(
            modified_linear(128 * 4 * 4, 256),
            nn.ReLU(),
            modified_linear(256, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        x = self.classifier(x)
        return x

    def prune(self,threshold):
        for module in self.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                module.prune(threshold)
    
    def quantize(self,k):
        for module in self.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                module.quantize(k)

In [48]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: cuda


In [49]:
import torch.optim as optim
from torch import nn
from tqdm import tqdm
import torch
def train_model(model, train_loader,device,epochs=5,lr=0.01, mode="Training"):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)

    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch [{epoch+1}/{epochs}] Loss: {running_loss/len(train_loader):.4f}")

def evaluate(model, loader , device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total


In [50]:
import torch
from torch import nn

def prune_model(model, prune_fraction):
    assert 0.0 < prune_fraction < 1.0
    all_weights = []
    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            w = module.weight.detach().abs().flatten()
            all_weights.append(w)
    all_weights = torch.cat(all_weights)

    k = int(prune_fraction * all_weights.numel())
    threshold = torch.kthvalue(all_weights, k).values.item()
    model.prune(threshold)
    return 

def quantize_model(model,K):
    model.quantize(K)
    return

In [51]:
model = SmallCIFARNet()
train_dataloader ,test_dataloader = CIFAR10_loader()
model.to(device)
train_model(model,train_dataloader,device,100)
print(evaluate(model,test_dataloader,device))
prune_model(model,0.9)
print(evaluate(model,test_dataloader,device))
train_model(model,train_dataloader,device,30)
print(evaluate(model,test_dataloader,device))
quantize_model(model,32)
print(evaluate(model,test_dataloader,device))
train_model(model,train_dataloader,device,30)
print(evaluate(model,test_dataloader,device))
torch.save(model.state_dict(), "model.pth")


Epoch [1/100] Loss: 2.3026
Epoch [2/100] Loss: 2.2984
Epoch [3/100] Loss: 2.0172
Epoch [4/100] Loss: 1.7654
Epoch [5/100] Loss: 1.5830
Epoch [6/100] Loss: 1.4367
Epoch [7/100] Loss: 1.3273
Epoch [8/100] Loss: 1.2366
Epoch [9/100] Loss: 1.1416
Epoch [10/100] Loss: 1.0634
Epoch [11/100] Loss: 0.9718
Epoch [12/100] Loss: 0.8968
Epoch [13/100] Loss: 0.8219
Epoch [14/100] Loss: 0.7348
Epoch [15/100] Loss: 0.6534
Epoch [16/100] Loss: 0.5760
Epoch [17/100] Loss: 0.5093
Epoch [18/100] Loss: 0.4382
Epoch [19/100] Loss: 0.3665
Epoch [20/100] Loss: 0.3038
Epoch [21/100] Loss: 0.2549
Epoch [22/100] Loss: 0.2257
Epoch [23/100] Loss: 0.1733
Epoch [24/100] Loss: 0.1646
Epoch [25/100] Loss: 0.1330
Epoch [26/100] Loss: 0.1110
Epoch [27/100] Loss: 0.0916
Epoch [28/100] Loss: 0.0814
Epoch [29/100] Loss: 0.0779
Epoch [30/100] Loss: 0.0701
Epoch [31/100] Loss: 0.0588
Epoch [32/100] Loss: 0.0472
Epoch [33/100] Loss: 0.0502
Epoch [34/100] Loss: 0.0392
Epoch [35/100] Loss: 0.0416
Epoch [36/100] Loss: 0.0408
E

In [52]:
torch.save(model.state_dict(), "/kaggle/working/models.pth")
!ls -lh /kaggle/working

total 22M
-rw-r--r-- 1 root root 1.3M Dec 15 16:09 compressed.npz
drwxr-xr-x 3 root root 4.0K Dec 15 15:46 data
-rw-r--r-- 1 root root  11M Dec 15 16:55 model.pth
-rw-r--r-- 1 root root  11M Dec 15 16:55 models.pth


In [53]:
import numpy as np

def is_compressed_layer(module):
    return isinstance(module, (modified_linear, modified_conv2d))

def export_compressed_model(model):
    """
    Returns:
        dict: layer_name -> compressed data
    """
    compressed = {}

    for name, module in model.named_modules():
        if is_compressed_layer(module):

            layer_data = {}

            # shape (for sanity / reconstruction)
            layer_data["shape"] = tuple(module.weight.shape)

            # mask (bool)
            layer_data["mask"] = (
                module.mask.detach()
                .cpu()
                .numpy()
                .astype(np.bool_)
            )

            # assignments (cluster indices)
            layer_data["assignments"] = (
                module.assignments.detach()
                .cpu()
                .numpy()
                .astype(np.uint8)
            )

            # codebook (centroids)
            layer_data["codebook"] = (
                module.hashtable.detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            # bias (optional)
            if module.bias is not None:
                layer_data["bias"] = (
                    module.bias.detach()
                    .cpu()
                    .numpy()
                    .astype(np.float32)
                )
            else:
                layer_data["bias"] = None

            compressed[name] = layer_data

    return compressed

def save_model_npz(model, savepath):
    compressed = export_compressed_model(model)
    np.savez(savepath, **compressed)

save_model_npz(model,'compressed.npz')
import torch


In [54]:
import numpy as np
from scipy.sparse import csr_matrix

def load_csr_from_npz(npz_path):
    raw = np.load(npz_path, allow_pickle=True)
    csr_layers = {}

    for layer_name in raw.files:
        layer = raw[layer_name].item()

        mask = layer["mask"]
        assignments = layer["assignments"]
        shape = tuple(layer["shape"])
        codebook = layer["codebook"]
        bias = layer["bias"]

        # ---- HANDLE SHAPE ----
        if len(shape) == 2:
            # Linear layer
            mask_2d = mask
            assign_2d = assignments
            out_shape = shape

        elif len(shape) == 4:
            # Conv2d → flatten per output channel
            oc, ic, kh, kw = shape
            mask_2d = mask.reshape(oc, -1)
            assign_2d = assignments.reshape(oc, -1)
            out_shape = (oc, ic * kh * kw)

        else:
            raise ValueError(f"Unsupported shape: {shape}")

        # ---- CSR construction ----
        rows, cols = np.nonzero(mask_2d)
        values = assign_2d[rows, cols]

        csr = csr_matrix(
            (values, (rows, cols)),
            shape=out_shape
        )

        csr_layers[layer_name] = {
            "csr": csr,
            "codebook": codebook,
            "bias": bias,
            "orig_shape": shape
        }

    return csr_layers
csr_layers = load_csr_from_npz("compressed.npz")


print(csr_layers)

{'features.0': {'csr': <Compressed Sparse Row sparse matrix of dtype 'uint8'
	with 705 stored elements and shape (32, 27)>, 'codebook': array([ 0.31895843, -0.18846938, -0.4907256 ,  0.31130722,  0.61514145,
       -1.0477145 , -0.21135458,  0.17289683, -0.3482385 , -0.04168737,
       -0.601513  ,  0.70020586, -0.33341852,  0.09935004, -0.08160758,
        0.71602285, -0.35441446,  0.12218665, -0.2120142 ,  0.21299118,
       -0.03985472,  0.7818254 , -0.01742598, -0.20591974, -0.03608188,
        1.1634445 , -0.5682135 , -0.40503204, -0.4369205 , -0.09459506,
        0.38314092, -0.92297626], dtype=float32), 'bias': array([ 0.11851358, -0.03156817, -0.00664849,  0.02013937, -0.38571104,
       -0.02728759, -0.02212444, -0.03333478, -0.3539644 ,  0.08681335,
       -0.8816437 , -0.31458354, -0.45763344, -0.02531132,  0.52625734,
       -0.0881317 ,  0.07302954, -0.01665829,  0.01743345,  0.23042801,
       -0.1346    ,  0.39426774,  0.15555093,  0.21878827, -0.03651807,
        0.2839